In [1]:
import pandas as pd

In [2]:
posts = pd.read_csv("https://raw.githubusercontent.com/allatambov/PyPerm25/refs/heads/main/hseteachers.csv")

In [3]:
# отбираем столбцы

small = posts[["id", "date", "text", "likes", 
               "reposts", "views", "comments"]]

In [4]:
# выключаем предупреждения

pd.set_option('chained_assignment', None)

In [5]:
# внутри ячеек – словари, которые считались как текст
# по факту имеем дело с невалидными JSON-строками
# извлекаем из невалидных JSON-строк нужные числа после count
# с помощью регулярных выражений

pattern = "'count':\s(\d+)"

small["nlikes"] = small["likes"].str.extract(pattern).astype(int)
small["nreposts"] = small["reposts"].str.extract(pattern).astype(int)
small["ncomments"] = small["comments"].str.extract(pattern).astype(int)

In [6]:
# то же для числа просмотров, но столбец
# нельзя сделать типом int, так как есть пропуски NaN
# поэтому делаем float, заполняем пропуски 0, а потом уже 
# превращаем в int

small["nviews"] = small["views"].str.extract(pattern).astype(float)
small["nviews"] = small["nviews"].fillna(0).astype(int)

In [7]:
# удаляем старые столбцы

small.drop(columns = ["likes", "reposts", 
                      "comments", "views"], inplace = True)

In [8]:
# разбираемся с датой, из UNIX в date-time

small["date"] = pd.to_datetime(small["date"], unit = "s")

In [9]:
# превращаем date-time в строку и извлекаем год + день недели

small["year"] = small["date"].dt.strftime("%Y")
small["wday"] = small["date"].dt.strftime("%A")
print(small.head())

      id                date  \
0  40491 2025-04-18 13:22:37   
1  40489 2025-04-17 14:38:47   
2  40488 2025-04-12 12:24:34   
3  40487 2025-04-12 10:09:22   
4  40485 2025-04-05 12:11:36   

                                                text  nlikes  nreposts  \
0  Если вы общаетесь с Богом - это хорошо,\nА вот...      20         8   
1  Студенты: на лекции про революцию Мэйдзи в Япо...      30         2   
2  «С точки зрения Сергея Караганова: ядерный уда...      24        10   
3  Навалилось много дел по учебе, ничего не успев...       1         0   
4  Вообще, расцвет математики и матанализа пришел...      32        11   

   ncomments  nviews  year      wday  
0          1    1003  2025    Friday  
1          0    1888  2025  Thursday  
2          0    1998  2025  Saturday  
3          0    1114  2025  Saturday  
4          0    2585  2025  Saturday  


In [10]:
# как разбить текст на части?

print(small["text"][0])

Если вы общаетесь с Богом - это хорошо,
А вот если Бог начинает общаться с вами - это уже проблема.

#ВШЭСПБ #Юрфак #Закревский_ВШЭ


In [12]:
# split() с expand растягивает полученный список строк
# на отдельные столбцы

new = small["text"].str.split("#", expand = True)
new.head()

,0,1,2,3,4,5,6
0,"Если вы общаетесь с Богом - это хорошо,\nА вот...",ВШЭСПБ,Юрфак,Закревский_ВШЭ,None,None,None
1,Студенты: на лекции про революцию Мэйдзи в Япо...,Суздальцев_ВШЭ,ПЭИ,None,None,None,None
2,«С точки зрения Сергея Караганова: ядерный уда...,Суслов_ВШЭ,Введение_в_МО,None,None,None,None
3,"Навалилось много дел по учебе, ничего не успев...",None,None,None,None,None,None
4,"Вообще, расцвет математики и матанализа пришел...",Лебедев_ВШЭ,МИЭМ,None,None,None,None


In [13]:
# iloc – выбор строк/ столбцов по индексам
# забираем первые 3, переименовываем

add = new.iloc[:, 0:3]
add.columns = ["phrase", "name", "place"]

In [15]:
# итог – склеиваем оба датафрейма по столбцам

final = pd.concat([small, add], axis = 1)
final.head()

,id,date,text,nlikes,nreposts,ncomments,nviews,year,wday,phrase,name,place
0,40491,2025-04-18 13:22:37,"Если вы общаетесь с Богом - это хорошо,\nА вот...",20,8,1,1003,2025,Friday,"Если вы общаетесь с Богом - это хорошо,\nА вот...",ВШЭСПБ,Юрфак
1,40489,2025-04-17 14:38:47,Студенты: на лекции про революцию Мэйдзи в Япо...,30,2,0,1888,2025,Thursday,Студенты: на лекции про революцию Мэйдзи в Япо...,Суздальцев_ВШЭ,ПЭИ
2,40488,2025-04-12 12:24:34,«С точки зрения Сергея Караганова: ядерный уда...,24,10,0,1998,2025,Saturday,«С точки зрения Сергея Караганова: ядерный уда...,Суслов_ВШЭ,Введение_в_МО
3,40487,2025-04-12 10:09:22,"Навалилось много дел по учебе, ничего не успев...",1,0,0,1114,2025,Saturday,"Навалилось много дел по учебе, ничего не успев...",None,None
4,40485,2025-04-05 12:11:36,"Вообще, расцвет математики и матанализа пришел...",32,11,0,2585,2025,Saturday,"Вообще, расцвет математики и матанализа пришел...",Лебедев_ВШЭ,МИЭМ
